Importación de módulos necesarios

**OCR** (Lectura de matrículas)

In [75]:
import time
from pathlib import Path

import cv2
import easyocr
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor
from ultralytics import RTDETR


Carga de modelos

In [ ]:
val_images_dir = Path("./dataset/val/images")
plate_model = RTDETR("./runs/detect/rtdetr_runs/plates_detection/weights/best.pt")

# OCR: EasyOCR
easy_reader = easyocr.Reader(['es', 'en'], gpu=torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

# VLM: SmolVLM2
model_id = "HuggingFaceTB/SmolVLM2-500M-Instruct"
proccessor = AutoProcessor.from_pretrained(model_id)
vlm_model = (
    AutoModelForImageTextToText.from_pretrained(
        model_id,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        _attn_implementation="flash_attention_2"
        if torch.cuda.is_available()
        else "eager",
    )
    .to(device)
    .eval()
)


[2026-11-11 21:32:36,410] [ WARNING] easyocr.py:71 - Using CPU. Note: This module is much faster with a GPU.


Funciones de inferencia

In [82]:
def infer_easyocr(images):
    """Sustituto con EasyOCR que lee directamente sobre el recorte en memoria."""
    start = time.time()
    
    h, w = images.shape[:2]
    if h > 0 and w > 0:
        scaled_img = cv2.resize(images, (w * 2, h * 2), interpolation=cv2.INTER_CUBIC)
    else:
        scaled_img = images
        
    result = easy_reader.readtext(scaled_img, detail=0)
    text = "".join(result).replace(" ", "").upper()
    end = time.time()
    return text, (end - start) * 1000


def infer_smolvlm2(images):
    """Función de inferencia para el VLM: SmolVLM2."""
    img_rgb = cv2.cvtColor(images, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(img_rgb)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": "Extract all the text visible in this image. Reply ONLY with the text, without any extra words.",
                },
            ],
        }
    ]

    start = time.time()
    prompt = proccessor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = proccessor(text=prompt, images=[image], return_tensors="pt").to(device)

    generated_ids = vlm_model.generate(**inputs, max_new_tokens=20)
    generated_texts = proccessor.batch_decode(generated_ids, skip_special_tokens=True)

    text = generated_texts[0].split("Assistant:")[-1].replace(" ", "").strip().upper()
    end = time.time()
    return text, (end - start) * 1000

Ejecución

In [83]:
results = []
image_files = list(val_images_dir.glob("*.jpg"))

for img_path in image_files:
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue

    res = plate_model.predict(frame, conf=0.3, verbose=False)[0]

    if len(res.boxes):
        i = int(res.boxes.conf.argmax())
        px1, py1, px2, py2 = map(int, res.boxes.xyxy[i].cpu().numpy())

        # Recorte en memoria
        crop_matricula = frame[py1:py2, px1:px2]

        if crop_matricula.size > 0:
            text_easy, time_easy = infer_easyocr(crop_matricula)
            text_smol, time_smol = infer_smolvlm2(crop_matricula)
            
            results.append(
                {
                    "Imagen": img_path.name,
                    "Texto_EasyOCR": text_easy,
                    "Tiempo_EasyOCR_ms": time_easy,
                    "Texto_SmolVLM2": text_smol,
                    "Tiempo_SmolVLM2_ms": time_smol,
                }
    )

Cálculo de las métricas

In [84]:
df_results = pd.DataFrame(results)

df_results["Respondio_EasyOCR"] = (df_results["Texto_EasyOCR"].str.strip() != "").astype(int)
df_results["Respondio_SmolVLM2"] = (df_results["Texto_SmolVLM2"].str.strip() != "").astype(int)

comparativa = pd.DataFrame(
    {
        "Modelo": ["EasyOCR", "SmolVLM2"],
        "Tiempo Inferencia Medio (ms)": [
            df_results["Tiempo_EasyOCR_ms"].mean(),
            df_results["Tiempo_SmolVLM2_ms"].mean(),
        ],
        "Tasa de Lectura Exitosa (%)": [
            df_results["Respondio_EasyOCR"].mean() * 100,
            df_results["Respondio_SmolVLM2"].mean() * 100,
        ],
    }
).set_index("Modelo")

df_results.to_csv("comparativa_ocr_no_labels.csv", index=False)
display(comparativa.round(3))

,Tiempo Inferencia Medio (ms),Tasa de Lectura Exitosa (%)
Modelo,,
EasyOCR,79.072,100.0
SmolVLM2,6322.811,100.0


Generamos el vídeo a partir del vídeo sin anonimizar (RT-DETR)

In [80]:
# Configuración de archivos de entrada y salida
video_input = "output_rtdetr.mp4"
csv_input = "detections_rtdetr.csv"

video_output = "output_rtdetr_ocr.avi"
csv_output = "detections_rtdetr_ocr.csv"

df = pd.read_csv(csv_input)

if "texto_matricula" not in df.columns:
    df["texto_matricula"] = ""

vid = cv2.VideoCapture(video_input)
fps = vid.get(cv2.CAP_PROP_FPS)
w = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(video_output, cv2.VideoWriter_fourcc(*"XVID"), fps, (w, h))

frame_n = 0

while True:
    ret, frame = vid.read()
    if not ret:
        break
    frame_n += 1

    # Filtrar las detecciones que corresponden a este fotograma exacto
    detecciones_frame = df[df["fotograma"] == frame_n]

    for index, row in detecciones_frame.iterrows():
        if row["matricula_en_su_caso"] == "Si":
            mx1 = max(0, int(row["mx1"]))
            my1 = max(0, int(row["my1"]))
            mx2 = min(w, int(row["mx2"]))
            my2 = min(h, int(row["my2"]))

            # Recortar la región de la matrícula
            crop_matricula = frame[my1:my2, mx1:mx2]

            if crop_matricula.size > 0:
                texto_final, _ = infer_easyocr(crop_matricula)

                # Guardar el texto en el DataFrame
                df.at[index, "texto_matricula"] = texto_final

                # Visualización en el fotograma
                cv2.putText(
                    frame,
                    texto_final,
                    (mx1, my1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 0),
                    2,
                )

    out.write(frame)

vid.release()
out.release()

df.to_csv(csv_output, index=False, encoding="utf-8")

print("Proceso completado")
print(f"Vídeo con textos OCR guardado en: {video_output}")
print(f"CSV actualizado guardado en: {csv_output}")

Proceso completado
Vídeo con textos OCR guardado en: output_rtdetr_ocr.avi
CSV actualizado guardado en: detections_rtdetr_ocr.csv


[mov,mp4,m4a,3gp,3g2,mj2 @ 0x55e86fe33c80] moov atom not found
[ERROR:0@1488.422] global /io/opencv/modules/videoio/src/cap.cpp (595) open VIDEOIO(CV_IMAGES): raised OpenCV exception:

OpenCV(4.6.0) /io/opencv/modules/videoio/src/cap_images.cpp:253: error: (-5:Bad argument) CAP_IMAGES: can't find starting number (in the name of file): output_rtdetr_ocr.avi in function 'icvExtractPattern'


